In [ ]:
pip install keras-tuner

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 KB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Trans-Ubiquitination-Colab/BraveHeart

/content/drive/MyDrive/Trans-Ubiquitination-Colab/BraveHeart


In [ ]:
import numpy as np
x = np.load("homo_cro_embeddings_X.npy")

In [ ]:
x.shape

(8244, 17664)

In [ ]:
y = np.load("homo_cro_embeddings_Y.npy")

In [ ]:
y.shape

(8244,)

In [ ]:
import pandas as pd
x_dataframe = pd.DataFrame(x)

In [ ]:
x_dataframe

,0,1,2,3,4,5,6,7,8,9,...,17654,17655,17656,17657,17658,17659,17660,17661,17662,17663
0,-0.796074,-0.155738,-0.188806,0.012723,-0.334955,-0.083902,0.495496,0.169278,-0.145961,-0.567864,...,0.057337,0.027197,-0.198353,-0.237782,0.055384,-0.085902,0.012321,0.056786,-0.244190,-0.042030
1,-0.299266,0.759266,-0.855120,-1.268924,-0.739080,0.115522,0.714832,-0.474045,-0.029525,0.756448,...,-0.120423,-0.051242,0.029189,-0.154974,0.243588,-0.262271,-0.140798,0.252195,-0.218730,0.528034
2,-0.750504,-0.443141,-0.182078,0.024726,-0.150807,0.020198,0.479671,0.294284,-0.207846,-0.362574,...,0.047228,-0.012126,-0.179511,-0.185440,0.032311,-0.094223,-0.003469,0.057470,-0.233205,-0.037691
3,-0.271679,1.037377,-1.113603,-1.461074,-1.045115,0.090262,1.080848,-0.200772,-0.187137,0.268036,...,-0.119162,0.130893,0.057593,-0.013464,0.685510,-0.319512,-0.394472,0.248184,-0.317357,0.788031
4,-0.690989,-0.202762,-0.295541,-0.124270,-0.433634,-0.150932,0.657722,0.162637,-0.073882,-0.489217,...,0.056641,0.018768,-0.165575,-0.202661,0.042927,-0.076864,-0.045153,0.070326,-0.245810,-0.053473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8239,-0.076418,0.840784,-0.926488,-1.131922,-1.155548,0.033180,1.129933,-0.430581,-0.475102,-0.467268,...,0.136353,0.043245,0.063142,0.085686,0.371630,-0.293534,-0.278110,0.024744,-0.286848,0.354133
8240,-0.620570,-0.255833,-0.313391,0.126946,-0.399787,-0.011806,0.456605,0.163411,-0.111855,-0.380473,...,0.045922,-0.011741,-0.204108,-0.201030,0.055292,-0.100267,-0.039191,0.070281,-0.230424,-0.068450
8241,-0.021300,1.171911,-1.207966,-1.463888,-1.177899,-0.004736,1.383463,-0.075069,0.106526,-0.546634,...,-0.011968,0.235013,-0.156180,0.297178,0.778926,-0.303521,-0.525506,0.179764,-0.038277,0.945448
8242,-0.736349,-0.284337,-0.140450,0.063674,-0.528563,0.183587,0.490671,0.248182,-0.199043,-0.315709,...,0.045766,0.009356,-0.183824,-0.219212,0.043026,-0.092335,-0.009064,0.107981,-0.228210,-0.083193


In [ ]:
y

array([1, 1, 1, ..., 0, 0, 0])

# 1D CNN

In [ ]:
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Activation, Conv1D, ZeroPadding1D, MaxPooling1D, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras import utils
import tensorflow as tf
from tensorflow import keras
from kerastuner import RandomSearch
from tensorflow.keras import utils
from sklearn.metrics import precision_recall_curve

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.20, random_state = 42)
xval, xtest, yval, ytest = train_test_split(xtest, ytest, test_size = 0.5)

def CNN_1D():
    model = Sequential()

    # layer 1
    model.add(Conv1D(8, 3, input_shape=(23*768, 1), activation="relu"))
    model.add(MaxPooling1D(2))
    model.add(Dropout(0.1))

    # layer 2
    model.add(Conv1D(16, 3, activation="relu"))
    model.add(MaxPooling1D(2))
    model.add(Dropout(0.2))

    # Flattening Layer:
    model.add(Flatten())
    model.add(Dense(64, activation="relu"))

    # Last Layer:
    model.add(Dense(2, activation="softmax"))
    model.compile(loss="categorical_crossentropy", optimizer=Adam(lr = 0.00005),
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model

model1 = CNN_1D()

a = np.asarray(xtrain).reshape(len(np.asarray(xtrain)),23*768,1)

history1_ = model1.fit(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),23*768,1), utils.to_categorical(ytrain,2),
                    validation_data=(np.asarray(xval).reshape(len(np.asarray(xval)),23*768,1), utils.to_categorical(yval,2)),
                    epochs=50, batch_size=20, verbose=1)    # epoch = 25   batch_size=20

score = model1.evaluate(np.asarray(xtest).reshape(len(np.asarray(xtest)),23*768,1), utils.to_categorical(ytest,2), verbose=1)
print("categorical_crossentropy loss, Accuracy, Precision, Recall scores:", score)


plt.plot(history1_.history["accuracy"])
plt.plot(history1_.history["val_accuracy"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("True positive rate", fontsize = 15)
plt.title("1D CNN Model Accuracy", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()
plt.figure(figsize=(10,10), dpi = 600)

plt.plot(history1_.history["loss"])
plt.plot(history1_.history["val_loss"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Loss", fontsize = 15)
plt.title("1D CNN Model Loss", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()

print(model1.summary())

import seaborn as sns
from sklearn.metrics import confusion_matrix
cnn_predictions1 = model1.predict(xtest)
bert2_probs_cnn1 = cnn_predictions1[:,1]
cnn_predictions1 = np.argmax(cnn_predictions1, axis = 1)
confusion_matrix = confusion_matrix(ytest, cnn_predictions1)
sns.heatmap(confusion_matrix, annot = True, fmt = "d", cbar = False)
plt.title("BERT + 1D CNN Confusion Matrix", fontsize = 20)

plt.show()

from sklearn.metrics import roc_curve
fpr_keras_bert_cnn1_hyp, tpr_keras_bert_cnn1_hyp, _ = roc_curve(ytest, bert2_probs_cnn1)

from sklearn.metrics import auc
auc_keras_bert_cnn1_hyp = auc(fpr_keras_bert_cnn1_hyp, tpr_keras_bert_cnn1_hyp)
print("AUC Score", auc_keras_bert_cnn1_hyp)split
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_keras_bert_cnn1_hyp, tpr_keras_bert_cnn1_hyp, label="BERT + 1D-CNN: {:.3f}".format(auc_keras_bert_cnn1_hyp))
plt.xlabel("False positive rate", fontsize = 15)
plt.ylabel("True positive rate", fontsize = 15)
plt.title("ROC curve for BERT + 1D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

print("----------------------------------------------")


precision_bert2_cnn1, recall_bert2_cnn1, _ = precision_recall_curve(ytest, bert2_probs_cnn1)
auc_precision_recall_bert2_cnn1 = auc(recall_bert2_cnn1, precision_bert2_cnn1)

plt.plot([0, 1], [0, 1], 'k--')
plt.plot(recall_bert2_cnn1, precision_bert2_cnn1, label="BERT + 1D-CNN: {:.3f}".format(auc_precision_recall_bert2_cnn1))
plt.xlabel("Recall", fontsize = 15)
plt.ylabel("Precision", fontsize = 15)
plt.title("PR curve for BERT + 1D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

print("----------------------------------------------")

# 2D CNN Without Tuner

In [ ]:
################ Without tuner ########################
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Activation, MaxPooling1D, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras import utils
import tensorflow as tf
from tensorflow import keras
from kerastuner import RandomSearch
from tensorflow.keras import utils
from sklearn.metrics import precision_recall_curve



xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.20, random_state=None, shuffle=True)
xval, xtest, yval, ytest = train_test_split(xtest, ytest, test_size = 0.5, random_state=None, shuffle=True)

def CNN_2D():
    model = Sequential()

    # layer 1
    model.add(Conv2D(32, 4, 4, input_shape=(768, 23, 1), activation="relu")) 
    model.add(MaxPooling2D(2))
    model.add(Dropout(0.1))


    # layer 2
    model.add(Conv2D(16, 2, 2, activation="relu"))
    #model.add(MaxPooling2D(2))
    model.add(Dropout(0.2))

    

    # Flattening Layer:
    model.add(Flatten())
    model.add(Dense(96, activation="relu"))
    model.add(Dense(32, activation="relu"))

    # Last Layer:
    model.add(Dense(2, activation="softmax"))
    model.compile(loss="binary_crossentropy", optimizer=Adam(lr = 0.001),
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model

model2 = CNN_2D()

stop_early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0.0008, patience=40,  verbose=1, mode='min')

history2_ = model2.fit(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),768,23,1), utils.to_categorical(ytrain,2),
                    validation_data=(np.asarray(xval).reshape(len(np.asarray(xval)),768,23,1), utils.to_categorical(yval,2)), callbacks = [stop_early],
                    epochs=200, batch_size=150, verbose=1)


<ipython-input-13-4a9415e3f607>:13: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner import RandomSearch
/usr/local/lib/python3.8/dist-packages/keras/optimizers/optimizer_v2/adam.py:110: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(Adam, self).__init__(name, **kwargs)


Epoch 1/200
44/44 [==============================] - 10s 26ms/step - loss: 0.6917 - accuracy: 0.5210 - precision: 0.5210 - recall: 0.5210 - val_loss: 0.6849 - val_accuracy: 0.5959 - val_precision: 0.5959 - val_recall: 0.5959
Epoch 2/200
44/44 [==============================] - 1s 14ms/step - loss: 0.6819 - accuracy: 0.5629 - precision: 0.5629 - recall: 0.5629 - val_loss: 0.6627 - val_accuracy: 0.6189 - val_precision: 0.6189 - val_recall: 0.6189
Epoch 3/200
44/44 [==============================] - 1s 13ms/step - loss: 0.6674 - accuracy: 0.5970 - precision: 0.5970 - recall: 0.5970 - val_loss: 0.6437 - val_accuracy: 0.6638 - val_precision: 0.6638 - val_recall: 0.6638
Epoch 4/200
44/44 [==============================] - 1s 14ms/step - loss: 0.6596 - accuracy: 0.6026 - precision: 0.6026 - recall: 0.6026 - val_loss: 0.6407 - val_accuracy: 0.6444 - val_precision: 0.6444 - val_recall: 0.6444
Epoch 5/200
44/44 [==============================] - 1s 13ms/step - loss: 0.6464 - accuracy: 0.6249 - p

In [ ]:
model2.evaluate(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1), utils.to_categorical(ytest,2))

26/26 [==============================] - 0s 5ms/step - loss: 0.6613 - accuracy: 0.6509 - precision: 0.6509 - recall: 0.6509


[0.6613085865974426,
 0.6509090662002563,
 0.6509090662002563,
 0.6509090662002563]

In [ ]:
plt.clf()
plt.plot(history2_.history["accuracy"])
plt.plot(history2_.history["val_accuracy"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Accuracy", fontsize = 15)
plt.title("2D CNN Model Accuracy", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()


plt.clf()
plt.plot(history2_.history["loss"])
plt.plot(history2_.history["val_loss"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Loss", fontsize = 15)
plt.title("2D CNN Model Loss", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()


print(model2.summary())

plt.clf()
import seaborn as sns
from sklearn.metrics import confusion_matrix
cnn_predictions3 = model2.predict(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1)) 
bert2_probs = cnn_predictions3[:,1]
print(cnn_predictions3[:,1])   
cnn_predictions3 = np.argmax(cnn_predictions3, axis = 1)
print(cnn_predictions3)
confusion_matrix = confusion_matrix(ytest, cnn_predictions3)
sns.heatmap(confusion_matrix, annot = True, fmt = "d", cbar = False)
plt.title("BERT + 2D CNN + H. Tuning Confusion Matrix", fontsize = 20)
plt.show()




from sklearn.metrics import roc_curve
fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp, thresholds_keras = roc_curve(ytest, bert2_probs)

from sklearn.metrics import auc
auc_keras_bert_cnn2_hyp = auc(fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp)
print("AUC Score", auc_keras_bert_cnn2_hyp)

plt.clf()
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp, label="BERT + 2D-CNN: {:.3f}".format(auc_keras_bert_cnn2_hyp))
plt.xlabel("False positive rate", fontsize = 15)
plt.ylabel("True positive rate", fontsize = 15)
plt.title("ROC curve for BERT + 2D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()


precision_bert2, recall_bert2, _ = precision_recall_curve(ytest, bert2_probs)
auc_precision_recall_bert2 = auc(recall_bert2, precision_bert2)

plt.clf()
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(recall_bert2, precision_bert2, label="BERT + 2D-CNN: {:.3f}".format(auc_precision_recall_bert2))
plt.xlabel("Recall", fontsize = 15)
plt.ylabel("Precision", fontsize = 15)
plt.title("PR curve for BERT + 2D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()


print("----------------------------------------------")

# 2D CNN with Tuner

In [ ]:
######################################## BERT + 2D CNN PART ############################################

#x = np.load("homo_ubi_embeddings20000_1_X.npy")
#y = np.load("homo_ubi_embeddings20000_1_Y.npy")

def build_model(hp):
    # create model object
    model = keras.Sequential([

        # adding first convolutional layer
        keras.layers.Conv2D(
            # adding filter
            filters=hp.Int('conv_1_filter', min_value=8, max_value=32, step=2),
            # adding filter size or kernel size
            kernel_size=hp.Choice('conv_1_kernel', values=[2, 6]),
            # activation function
            activation='relu',
            input_shape=(768, 23, 1)),
        
        # adding second convolutional layer
        keras.layers.Conv2D(
            # adding filter
            filters=hp.Int('conv_2_filter', min_value=10, max_value=32, step=4),
            # adding filter size or kernel size
            kernel_size=hp.Choice('conv_2_kernel', values=[2, 8]),
            # activation function
            activation='relu'
        
        ),
        
        # adding flatten layer
        keras.layers.Flatten(),
        # adding dense layer
        keras.layers.Dense(
            units=hp.Int('dense_1_units', min_value=64, max_value=128, step=4),
            activation='relu'
        ),

        # adding flatten layer
        keras.layers.Flatten(),
        # adding dense layer
        keras.layers.Dense(
            units=hp.Int('dense_2_units', min_value=16, max_value=96, step=4),
            activation='relu'
        ),

        # output layer
        keras.layers.Dense(2, activation='softmax')
    ])

    # compilation of model
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', values=[0.00006, 0.00008, 0.00003, 0.00001])),  
                  loss='categorical_crossentropy',
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

    model.summary()
    return model

#importing random search
from kerastuner import RandomSearch
from kerastuner.engine.hyperparameters import HyperParameters

#creating randomsearch object
tuner = RandomSearch(build_model, objective='val_accuracy', max_trials = 20, directory = "output", project_name = "Crot_AfterHyperParameterTuning_2D_CNN")


xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.20, random_state=None, shuffle=True)
xval, xtest, yval, ytest = train_test_split(xtest, ytest, test_size = 0.5, random_state=None, shuffle=True)

stop_early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0.0008, patience=40,  verbose=1, mode='min')

tuner.search(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),768,23,1), utils.to_categorical(ytrain,2), 
             validation_data = (np.asarray(xval.reshape(len(np.asarray(xval)),768,23,1)), utils.to_categorical(yval,2)), epochs = 100, callbacks = [stop_early])


model_2D_ht = tuner.get_best_models(num_models=1)[0]

model_2D_ht.summary()

history3 = model_2D_ht.fit(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1), utils.to_categorical(ytest,2),
                           epochs=200, batch_size = 10,
                           initial_epoch=1)     # epochs=15

In [ ]:
history_dict = history2_.history
print(history_dict.keys())

dict_keys(['loss', 'accuracy', 'precision_2', 'recall_2', 'val_loss', 'val_accuracy', 'val_precision_2', 'val_recall_2'])


In [ ]:
plt.plot(history3.history["accuracy"])
plt.plot(history3.history["val_accuracy"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Accuracy", fontsize = 15)
plt.title("2D CNN Model Accuracy", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()
plt.figure(figsize=(10,10), dpi = 600)

plt.plot(history3.history["loss"])
plt.plot(history3.history["val_loss"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Loss", fontsize = 15)
plt.title("2D CNN Model Loss", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()

print(model_2D_ht.summary())

import seaborn as sns
from sklearn.metrics import confusion_matrix
cnn_predictions3 = model_2D_ht.predict(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1)) 
bert2_probs = cnn_predictions3[:,1]
print(cnn_predictions3[:,1])    
cnn_predictions3 = np.argmax(cnn_predictions3, axis = 1)
print(cnn_predictions3)
confusion_matrix = confusion_matrix(ytest, cnn_predictions3)
sns.heatmap(confusion_matrix, annot = True, fmt = "d", cbar = False)
plt.title("BERT + 2D CNN + H. Tuning Confusion Matrix", fontsize = 20)
plt.show()

from sklearn.metrics import roc_curve
print(cnn_predictions3)
fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp, thresholds_keras = roc_curve(ytest, bert2_probs)

from sklearn.metrics import auc
auc_keras_bert_cnn2_hyp = auc(fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp)
print("AUC Score", auc_keras_bert_cnn2_hyp)

plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp, label="BERT + 2D-CNN: {:.3f}".format(auc_keras_bert_cnn2_hyp))
plt.xlabel("False positive rate", fontsize = 15)
plt.ylabel("True positive rate", fontsize = 15)
plt.title("ROC curve for BERT + 2D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

print("----------------------------------------------")


precision_bert2, recall_bert2, _ = precision_recall_curve(ytest, bert2_probs)
auc_precision_recall_bert2 = auc(recall_bert2, precision_bert2)

#plt.clf()
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(recall_bert2, precision_bert2, label="BERT + 2D-CNN: {:.3f}".format(auc_precision_recall_bert2))
plt.xlabel("Recall", fontsize = 15)
plt.ylabel("Precision", fontsize = 15)
plt.title("PR curve for BERT + 2D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

print("----------------------------------------------")

# ViT

In [ ]:
pip install tensorflow-addons

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.1 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Activation, Conv1D, ZeroPadding1D, MaxPooling1D, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras import utils
import tensorflow as tf
from tensorflow import keras
from kerastuner import RandomSearch
from tensorflow.keras import utils
from sklearn.metrics import precision_recall_curve
from tensorflow.keras import layers
import tensorflow_addons as tfa
import tensorflow_addons as tfa
import pickle
import time
from tensorflow.keras.callbacks import TensorBoard
import keras_tuner as kt
from keras_tuner import RandomSearch


num_classes = 2
input_shape = (768, 23, 1)

In [ ]:
learning_rate = 0.001         # 0.001
weight_decay = 0.0001
batch_size = 200   # 256
num_epochs = 50                                                                        # num_epochs = 10
image_size = 16  # We'll resize input images to this size  # 72
patch_size = 6  # Size of the patches to be extract from the input images  # 6
num_patches = (image_size // patch_size) ** 2
projection_dim = 64
num_heads = 12     # 16
transformer_units = [projection_dim * 2, projection_dim, ]  # Size of the transformer layers
transformer_layers =  1     # 16  
mlp_head_units = [2048, 1024]  # Size of the dense layers of the final classifier

In [ ]:
import keras

#####  Use data augmentation

data_augmentation = keras.Sequential(
    [
        layers.Normalization(),
        layers.Resizing(image_size, image_size),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(factor=0.02),
        layers.RandomZoom(height_factor=0.2, width_factor=0.2),
    ],
    name="data_augmentation",
)
# Compute the mean and the variance of the training data for normalization.
data_augmentation.layers[0].adapt(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),768,23,1))

In [ ]:
#######  Implement multilayer perceptron (MLP)

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [ ]:
#######  Implement patch creation as a layer

class Patches(layers.Layer):
    def __init__(self, patch_size):
        super(Patches, self).__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

In [ ]:
######  Implement the patch encoding layer

class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super(PatchEncoder, self).__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

In [ ]:
# ################################ EDITTING VIT CODE #################################

######  Build the ViT model

def optimal_vit_classifier(hp):
    print("OK-1")
    inputs = layers.Input(shape=input_shape)
    # Augment data.
    augmented = data_augmentation(inputs)
    print(augmented.shape)
    # Create patches.
    patches = Patches(patch_size)(augmented)
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)
    print("OK-2")
 

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):                                                       
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads = num_heads, key_dim=projection_dim, dropout=0.1                             
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])
        print("OK-3")

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.5)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    # Classify outputs.
    logits = layers.Dense(num_classes, activation ="softmax")(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)
    print("OK-4")

    optimizer = tfa.optimizers.AdamW(learning_rate = learning_rate, weight_decay = weight_decay)   

    print("OK-5")
    
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', values=[0.0005, 0.0001])),    
                  loss='categorical_crossentropy',
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC(), 
                           tf.keras.metrics.AUC(curve = "ROC"), tf.keras.metrics.AUC(curve = "PR")])
    
    return model

tuner_ch = kt.RandomSearch(optimal_vit_classifier,
                        objective="val_accuracy",                                                                  
                        max_trials = 15, directory = "output", project_name = "Crot_AfterHyperParameterTuning_ViT_homo")


OK-1
(None, 16, 16, 1)
OK-2
OK-3
OK-4
OK-5


In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0.0008, patience=100,  verbose=1, mode='min')

tuner_ch.search(np.asarray(xtrain.reshape(len(np.asarray(xtrain)),768,23,1)), utils.to_categorical(ytrain,2),
                validation_data = (np.asarray(xval.reshape(len(np.asarray(xval)),768,23,1)), utils.to_categorical(yval,2)), epochs = 50, callbacks = [stop_early])

In [ ]:
model3 = tuner_ch.get_best_models(num_models=1)[0]
history3_ = model3.fit(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1), utils.to_categorical(ytest,2), epochs = 700)

OK-1
(None, 16, 16, 1)
OK-2
OK-3
OK-4
OK-5
Epoch 1/700
26/26 [==============================] - 4s 24ms/step - loss: 0.6726 - accuracy: 0.5927 - precision: 0.5927 - recall: 0.5927 - auc: 0.6198 - auc_1: 0.6198 - auc_2: 0.6029
Epoch 2/700
26/26 [==============================] - 1s 23ms/step - loss: 0.7098 - accuracy: 0.5685 - precision: 0.5685 - recall: 0.5685 - auc: 0.5702 - auc_1: 0.5702 - auc_2: 0.5461
Epoch 3/700
26/26 [==============================] - 1s 23ms/step - loss: 0.6721 - accuracy: 0.5952 - precision: 0.5952 - recall: 0.5952 - auc: 0.6215 - auc_1: 0.6215 - auc_2: 0.6035
Epoch 4/700
26/26 [==============================] - 1s 23ms/step - loss: 0.6847 - accuracy: 0.5527 - precision: 0.5527 - recall: 0.5527 - auc: 0.5769 - auc_1: 0.5769 - auc_2: 0.5673
Epoch 5/700
26/26 [==============================] - 1s 23ms/step - loss: 0.6698 - accuracy: 0.5794 - precision: 0.5794 - recall: 0.5794 - auc: 0.6246 - auc_1: 0.6246 - auc_2: 0.6105
Epoch 6/700
26/26 [=======================

'\ntuner.search(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),768,23,1), utils.to_categorical(ytrain,2), \n             validation_data = (np.asarray(xval.reshape(len(np.asarray(xval)),768,23,1)), utils.to_categorical(yval,2)), epochs = 100, callbacks = [stop_early])\n\n\nmodel_2D_ht = tuner.get_best_models(num_models=1)[0]\n\nmodel_2D_ht.summary()\n\nhistory3 = model_2D_ht.fit(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1), utils.to_categorical(ytest,2),\n                           epochs=200, batch_size = 100, validation_split=0.1,\n                           initial_epoch=1)     # epochs=15\n'